In [71]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [72]:
df = pd.read_csv('laptop_cleaned.csv')

In [73]:
df.head()

,Company,TypeName,Inches,Ram,OpSys,Weight,Price,TouchScreen,IPS,HD_4K,CPU_Brand,GPU_Brand,Memory_Type,CPU_Frequency,Length,Width,PPI
0,Apple,Ultrabook,13.3,8,macOS,0.862890,11.175769,0,1,0,Intel,Intel,SSD,2.3,2560.0,1600.0,5.429271
1,Apple,Ultrabook,13.3,8,macOS,0.850151,10.776798,0,0,0,Intel,Intel,Flash Storage,1.8,1440.0,900.0,4.857313
2,HP,Notebook,15.6,8,No OS,1.050822,10.329964,0,0,1,Intel,Intel,SSD,2.5,1920.0,1080.0,4.957319
3,Apple,Ultrabook,15.4,16,macOS,1.040277,11.814483,0,1,0,Intel,AMD,SSD,2.7,2880.0,1800.0,5.400579
4,Apple,Ultrabook,13.3,8,macOS,0.862890,11.473111,0,1,0,Intel,Intel,SSD,3.1,2560.0,1600.0,5.429271


In [74]:
X = df.drop(columns=['Price'])
y = df['Price']  # Already log-transformed

In [75]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# Save the test set to files
X_train.to_csv("splits/X_train.csv", index=False)
X_test.to_csv("splits/X_test.csv", index=False)
y_train.to_csv("splits/y_train.csv", index=False)
y_test.to_csv("splits/y_test.csv", index=False)

In [76]:
categorical_cols = ['Company', 'TypeName', 'OpSys', 'CPU_Brand', 'GPU_Brand', 'Memory_Type']
numerical_cols = [col for col in X.columns if col not in categorical_cols]

In [ ]:
### Why One-Hot Encoding?
We used One-Hot Encoding because it prevents the model from learning artificial ordinal relationships between categorical values.
Since Linear Regression treats input numerically, label encoding could mislead the model (e.g., Apple=0, Dell=1, MSI=2).
One-Hot ensures each category is treated independently, maintaining model fairness.

In [77]:
# One-Hot Encoding for categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ],
    remainder='passthrough'  # Pass numerical columns without change
)

# Create pipeline with Linear Regression
pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', LinearRegression())
])

In [78]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [79]:
# Predict on test set
y_pred = pipeline.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")


RMSE: 0.2485
R² Score: 0.8195


In [80]:
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100  # In %

# Adjusted R²
n = X_test.shape[0]
p = X_test.shape[1]
adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))

print(f"Adjusted R² Score: {adjusted_r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE: {mape:.2f}%")

Adjusted R² Score: 0.8077
MAE: 0.2032
MAPE: 1.88%


In [81]:
metrics = {
    "Metric": ["RMSE", "R²", "Adjusted R²", "MAE", "MAPE"],
    "Value": [rmse, r2, adjusted_r2, mae, mape]
}
results_df = pd.DataFrame(metrics)
print(results_df)


        Metric     Value
0         RMSE  0.248468
1           R²  0.819488
2  Adjusted R²  0.807651
3          MAE  0.203240
4         MAPE  1.880070


In [82]:
results_df.to_csv("lr_results_summary.csv", index=False)

In [83]:
joblib.dump(pipeline, 'linear_model_pipeline.pkl')

['linear_model_pipeline.pkl']

In [84]:
y_train_pred = pipeline.predict(X_train)
residuals_train = y_train - y_train_pred

# Test predictions
y_test_pred = pipeline.predict(X_test)
residuals_test = y_test - y_test_pred

print("Train residual mean:", residuals_train.mean())
print("Test residual mean:", residuals_test.mean())
print("Train residual std:", residuals_train.std())
print("Test residual std:", residuals_test.std())


Train residual mean: -1.505300469472573e-15
Test residual mean: -0.0014434101191756346
Train residual std: 0.25873766841110396
Test residual std: 0.24894112030733143


In [85]:
# Train metrics
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_train_pred)
mae_train = mean_absolute_error(y_train, y_train_pred)
mape_train = mean_absolute_percentage_error(y_train, y_train_pred) * 100

n_train = X_train.shape[0]
p_train = X_train.shape[1]
adjusted_r2_train = 1 - ((1 - r2_train) * (n_train - 1) / (n_train - p_train - 1))

print("\n=== Train Metrics ===")
print(f"Train RMSE: {rmse_train:.4f}")
print(f"Train R²: {r2_train:.4f}")
print(f"Train Adjusted R²: {adjusted_r2_train:.4f}")
print(f"Train MAE: {mae_train:.4f}")
print(f"Train MAPE: {mape_train:.2f}%")



=== Train Metrics ===
Train RMSE: 0.2586
Train R²: 0.8297
Train Adjusted R²: 0.8270
Train MAE: 0.2030
Train MAPE: 1.88%
